# Stage B validation — robustness + stress (§8.2.4 + §8.2.5)

Diagnostics that grade *how* the RF behaves, not just *whether* it scores well:
- **§8.2.4 robustness** — feature importances (do physically sensible predictors dominate?), AOD-binned residual envelope (does bias structure look like a classic MODIS-style low-bias-at-high-AOD?), and **CAMS prior bias attribution** (is the RF residual bias inherited from the CAMS background, or generated by the RF itself?).
- **§8.2.5 cloud-period recovery** — stress test against long occlusions.  Does the model invent plausible-but-wrong fills during multi-day cloud blackouts?

In [ ]:
from datetime import date
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
import validate as vb
import config as cfg

START = cfg.TEST_START
END   = cfg.TEST_END
print(f'Held-out window: {START} → {END}')

## §8.2.4a — Variable importance

Gini importances from the trained RF, normalised to percentages.  Sanity check: physically reasonable features (CAMS AOD, neighbour stats, time-of-day) should dominate.

In [ ]:
vi = vb.variable_importance("rf_default_residual")
vi

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
vi.iloc[::-1].plot.barh(x='feature', y='pct_importance', ax=ax, legend=False)
ax.set_xlabel('Gini importance (%)'); ax.set_title('RF predictor importance')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()

## §8.2.4b — Residual envelope vs AERONET AOD

Bins residuals (`sat − AERONET`) by AERONET AOD (0.1 width) and reports `n / bias / sigma` per bin.  Reveals AOD-dependent bias structure (Lee 2025 Fig. 7 analogue).

In [ ]:
pairs_rf   = vb.aeronet_pairs(START, END, candidate='rf',      blind_only=True)
pairs_krig = vb.aeronet_pairs(START, END, candidate='kriging', blind_only=True)
env      = vb.residual_envelope(pairs_rf)     # kept as `env` so §8.2.4c below still works
env_krig = vb.residual_envelope(pairs_krig)
env

In [ ]:
env_krig

In [ ]:
MIN_N_PLOT = 5  # hide bins with too few pairs to be readable

def _trim(e):
    return e[e['n'] >= MIN_N_PLOT] if not e.empty else e

er, ek = _trim(env), _trim(env_krig)

if not (er.empty and ek.empty):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    # small x-offset so errorbars don't sit on top of each other
    if not er.empty:
        ax.errorbar(er['aer_bin'] - 0.01, er['bias'], yerr=er['sigma'],
                    fmt='o-', capsize=3, lw=1.2, color='C0', label='RF')
    if not ek.empty:
        ax.errorbar(ek['aer_bin'] + 0.01, ek['bias'], yerr=ek['sigma'],
                    fmt='s--', capsize=3, lw=1.2, color='C3', label='ST-Kriging')
    ax.axhline(0, color='k', lw=0.6)
    ax.set_xlabel('AERONET AOD bin'); ax.set_ylabel('residual (sat − AERONET)')
    ax.set_title(f'Residual envelope vs AERONET AOD  (bins with n ≥ {MIN_N_PLOT})')
    ax.grid(alpha=0.3)
    ax.legend()

## §8.2.4c — CAMS prior bias attribution

The RF is trained on the residual `aod_minus_cams` ([config.py:160](config.py#L160)), so its output is `cams_aod + RF_residual_prediction`. Any systematic CAMS bias at a station propagates to the gap-fill unless the residual training saw enough local examples to push back.

If `cams_aod − AERONET` is already, say, +0.14 at Bac Lieu in the wet season, the residual-mode RF was **never** going to fix it — fixing it would require predicting a negative residual exactly when the regional training signal says the residual should be near zero.

This diagnostic decomposes the §8.2.4b envelope into two pieces on the same blind pairs:
- **CAMS bias** = `cams_aod − aer_aod` (the prior the RF stands on)
- **RF residual** = `rf_aod − cams_aod` (what the RF actually contributed)
- **Total** = `rf_aod − aer_aod` (what the user sees, == §8.2.4b)

If `Total ≈ CAMS bias` per (site, season), the RF is faithfully tracking CAMS and the bias is inherited — re-training won't help, swapping the prior or adding local correction will.

In [ ]:
import ancillary as anc
from validate import _site_rc

# Re-use the same blind pairs as §8.2.4b (pairs_rf) — no new I/O loop over slots,
# we just open each unique CAMS slot once and sample the station pixel.
site_rcs = {s: _site_rc(s) for s in pairs_rf['site'].unique()}

cams_vals = []
cams_cache: dict = {}
for slot_utc, site in pairs_rf[['slot_utc', 'site']].itertuples(index=False):
    grid = cams_cache.get(slot_utc)
    if grid is None:
        grid = anc.cams_aod_slot(slot_utc.to_pydatetime() if hasattr(slot_utc, 'to_pydatetime') else slot_utc)
        cams_cache[slot_utc] = grid
    if grid is None:
        cams_vals.append(np.nan)
        continue
    r, c = site_rcs[site]
    v = grid[r, c]
    cams_vals.append(float(v) if np.isfinite(v) else np.nan)

attr = pairs_rf[['slot_utc', 'site', 'season', 'aer_aod', 'sat_aod']].copy()
attr = attr.rename(columns={'sat_aod': 'rf_aod'})
attr['cams_aod']     = cams_vals
attr['cams_minus_aer'] = attr['cams_aod'] - attr['aer_aod']   # CAMS prior bias
attr['rf_residual']    = attr['rf_aod']   - attr['cams_aod']  # what RF added
attr['total_bias']     = attr['rf_aod']   - attr['aer_aod']   # == §8.2.4b residual

attr = attr.dropna(subset=['cams_aod'])
print(f'attributed pairs: {len(attr)}  (of {len(pairs_rf)} blind RF pairs)')
attr.head()

In [ ]:
MIN_N = 10  # drop (site, season) cells with too few blind pairs to trust

attribution = (
    attr.groupby(['site', 'season'])
        .agg(n=('aer_aod', 'size'),
             cams_bias =('cams_minus_aer', 'mean'),
             rf_added  =('rf_residual',    'mean'),
             total_bias=('total_bias',     'mean'))
        .reset_index()
)
attribution = attribution[attribution['n'] >= MIN_N]
attribution = attribution.reindex(
    attribution['total_bias'].abs().sort_values(ascending=False).index
).reset_index(drop=True)

attribution

## §8.2.5 — Cloud-period recovery stress test

Detects runs of ≥ 48 consecutive low-coverage slots (≈ 2.3 days of occlusion).  For each run, compares the gap-fill in the last cloudy slot against the first recovered slot (where the satellite is back) and reports `recovery_delta_rmse`.

In [ ]:
vb.cloud_period_recovery(START, END)